# 💰 FinTrack AI - Complete Backend Notebook

This notebook contains the complete backend of **FinTrack AI**, a personal finance manager that uses machine learning to:
- Categorise transactions automatically
- Predict future expenses
- Generate savings recommendations
- Provide spending insights
- Chat with users about finances
- Generate monthly PDF reports

**Author:** FinTrack AI Team  
**Version:** 1.0.0

## 📦 1. Install Dependencies

Run this cell to install all required packages.

In [4]:
pip install flask flask-cors pandas numpy reportlab

Note: you may need to restart the kernel to use updated packages.


## 🧠 2. Module: ml_engine.py

This module contains all AI/ML logic:
1. **Transaction Categorizer** - Rule-based keyword scoring
2. **Expense Predictor** - Linear regression on historical data
3. **Savings Recommender** - Budget analysis and savings tips
4. **Spending Insights** - Anomaly detection and pattern analysis
5. **AI Chatbot** - Rule-based financial assistant

In [5]:
"""
FinTrack AI - Machine Learning Engine
Handles: Transaction Categorization, Expense Prediction,
         Savings Recommendations, Spending Insights, AI Chatbot
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re
import json


# ─── 1. AI Transaction Categorizer ────────────────────────────────────────────

CATEGORY_KEYWORDS = {
    "Food & Dining": [
        "zomato", "swiggy", "restaurant", "cafe", "food", "pizza", "burger",
        "mcdonalds", "kfc", "dominos", "hotel", "dhaba", "biryani", "chai",
        "coffee", "snack", "lunch", "dinner", "breakfast", "meal", "eat",
        "kitchen", "tiffin", "dine", "eatery", "bakery", "sweet", "juice"
    ],
    "Shopping": [
        "amazon", "flipkart", "myntra", "meesho", "ajio", "nykaa", "market",
        "shop", "store", "purchase", "buy", "order", "clothes", "dress",
        "shoes", "bag", "watch", "accessory", "fashion", "retail", "mall",
        "bazaar", "grocery", "supermarket", "dmart", "reliance", "big bazaar"
    ],
    "Transportation": [
        "ola", "uber", "rapido", "auto", "cab", "taxi", "metro", "bus",
        "train", "railway", "irctc", "fuel", "petrol", "diesel", "parking",
        "toll", "flight", "indigo", "spicejet", "air india", "travel", "ride",
        "rickshaw", "bike", "vehicle", "transport"
    ],
    "Entertainment": [
        "netflix", "amazon prime", "hotstar", "spotify", "youtube", "movie",
        "cinema", "pvr", "inox", "bookmyshow", "concert", "event", "game",
        "gaming", "steam", "playstation", "subscription", "ott", "zee5",
        "disney", "hulu", "music", "show", "ticket","prime video"
    ],
    "Utilities": [
        "electricity", "water", "gas", "internet", "broadband", "wifi",
        "jio", "airtel", "vodafone", "bsnl", "mobile", "recharge", "bill",
        "utility", "maintenance", "rent", "housing", "electricity board",
        "bescom", "tata power", "dth", "tatasky"
    ],
    "Healthcare": [
        "hospital", "clinic", "doctor", "medicine", "pharmacy", "medical",
        "health", "lab", "test", "diagnostic", "apollo", "fortis",
        "1mg", "netmeds", "pharmeasy", "consultation", "surgery", "dental",
        "eye", "optician", "ayurveda", "homeopathy", "insurance"
    ],
    "Education": [
        "college", "university", "school", "course", "fee", "tuition",
        "book", "stationery", "pen", "notebook", "udemy", "coursera",
        "skill", "training", "coaching", "exam", "certification", "library",
        "assignment", "project", "study", "class", "workshop"
    ],
    "Travel": [
        "hotel", "resort", "oyo", "makemytrip", "goibibo", "booking",
        "airbnb", "yatra", "trip", "tour", "vacation", "holiday",
        "sightseeing", "trek", "adventure", "baggage"
    ],
}


def categorize_transaction(description: str, amount: float = 0, txn_type: str = "expense") -> str:
    """
    Rule-based + weighted keyword ML categorizer.
    Uses TF-style keyword scoring with amount heuristics.
    """
    if txn_type == "income":
        return "Income"

    desc_lower = description.lower()

    scores = {}
    for cat, keywords in CATEGORY_KEYWORDS.items():
        score = 0
        for kw in keywords:
            if kw in desc_lower:
                # Longer keyword matches score higher (more specific)
                score += len(kw.split())
        if score > 0:
            scores[cat] = score

    if not scores:
        # Amount-based fallback heuristics
        if amount < 200:
            return "Food & Dining"
        elif amount < 1000:
            return "Shopping"
        elif amount > 10000:
            return "Utilities"
        return "Other"

    return max(scores, key=scores.get)


# ─── 2. Expense Predictor (Linear Regression + Trend Analysis) ────────────────

def predict_future_expenses(df: pd.DataFrame) -> dict:
    """
    Predicts next 3 months of expenses per category
    using linear regression on historical monthly data.
    """
    if df.empty:
        return {"predictions": [], "total_predicted": 0}

    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.to_period("M")

    monthly = df.groupby(["month", "category"])["amount"].sum().reset_index()
    monthly["month_idx"] = monthly["month"].rank(method="dense").astype(int)

    predictions = []
    categories = monthly["category"].unique()
    now = datetime.now()

    for cat in categories:
        cat_data = monthly[monthly["category"] == cat].sort_values("month_idx")
        if len(cat_data) < 2:
            continue

        x = cat_data["month_idx"].values.reshape(-1, 1)
        y = cat_data["amount"].values

        # Simple least squares linear regression
        x_mean, y_mean = x.mean(), y.mean()
        beta = np.sum((x.flatten() - x_mean) * (y - y_mean)) / np.sum((x.flatten() - x_mean) ** 2)
        alpha = y_mean - beta * x_mean

        max_idx = cat_data["month_idx"].max()
        future_months = []
        for i in range(1, 4):
            pred_idx = max_idx + i
            pred_val = max(0, alpha + beta * pred_idx)
            month_label = (now + timedelta(days=30 * i)).strftime("%b %Y")
            future_months.append({"month": month_label, "predicted": round(pred_val, 2)})

        # Trend direction
        if beta > 50:
            trend = "↑ Rising"
        elif beta < -50:
            trend = "↓ Falling"
        else:
            trend = "→ Stable"

        predictions.append({
            "category": cat,
            "avg_monthly": round(float(y.mean()), 2),
            "trend": trend,
            "next_months": future_months,
            "confidence": min(95, max(60, int(70 + len(cat_data) * 5)))
        })

    # Total prediction for next month
    total_next = sum(
        p["next_months"][0]["predicted"]
        for p in predictions
        if p["next_months"]
    )

    return {
        "predictions": sorted(predictions, key=lambda x: x["avg_monthly"], reverse=True),
        "total_predicted_next_month": round(total_next, 2)
    }


# ─── 3. Savings Recommender ────────────────────────────────────────────────────

def generate_savings_recommendations(df: pd.DataFrame, df_budgets: pd.DataFrame) -> dict:
    """
    Analyzes spending patterns and generates personalized savings tips.
    """
    if df.empty:
        return {"recommendations": [], "potential_savings": 0}

    df["date"] = pd.to_datetime(df["date"])
    now = datetime.now()
    month_start = now.replace(day=1)

    expenses = df[(df["type"] == "expense") & (df["date"] >= month_start)]
    income_total = df[(df["type"] == "income") & (df["date"] >= month_start)]["amount"].sum()
    expense_total = expenses["amount"].sum()

    category_spend = expenses.groupby("category")["amount"].sum().to_dict()

    recommendations = []
    potential_savings = 0

    # Rule 1: Over-budget categories
    if not df_budgets.empty:
        for _, row in df_budgets.iterrows():
            spent = category_spend.get(row["category"], 0)
            if spent > row["monthly_limit"]:
                over = spent - row["monthly_limit"]
                potential_savings += over * 0.5
                recommendations.append({
                    "type": "budget_alert",
                    "priority": "high",
                    "icon": "⚠️",
                    "title": f"Over Budget: {row['category']}",
                    "detail": f"You've spent ₹{spent:,.0f} vs ₹{row['monthly_limit']:,.0f} budget. "
                              f"Cut by ₹{over:,.0f} to stay within limit.",
                    "saving": round(over * 0.5, 2)
                })

    # Rule 2: 50-30-20 rule analysis
    if income_total > 0:
        needs_pct = expense_total / income_total * 100
        savings_pct = (income_total - expense_total) / income_total * 100

        if savings_pct < 20:
            deficit = (income_total * 0.2) - (income_total - expense_total)
            potential_savings += deficit
            recommendations.append({
                "type": "savings_goal",
                "priority": "high",
                "icon": "🎯",
                "title": "Increase Savings Rate",
                "detail": f"You're saving {savings_pct:.1f}% of income. The 50-30-20 rule recommends "
                          f"saving 20%. Try to save ₹{deficit:,.0f} more this month.",
                "saving": round(deficit, 2)
            })

    # Rule 3: High entertainment / dining spend
    entertainment = category_spend.get("Entertainment", 0)
    food = category_spend.get("Food & Dining", 0)

    if entertainment > 2000:
        save_amt = entertainment * 0.3
        potential_savings += save_amt
        recommendations.append({
            "type": "category_tip",
            "priority": "medium",
            "icon": "🎬",
            "title": "Reduce Entertainment Spend",
            "detail": f"Entertainment at ₹{entertainment:,.0f}/month is high. "
                      f"Share subscriptions or use free alternatives to save ₹{save_amt:,.0f}.",
            "saving": round(save_amt, 2)
        })

    if food > 6000:
        save_amt = food * 0.25
        potential_savings += save_amt
        recommendations.append({
            "type": "category_tip",
            "priority": "medium",
            "icon": "🍽️",
            "title": "Optimize Food Spending",
            "detail": f"Food & Dining at ₹{food:,.0f}/month is above average. "
                      f"Cook at home 3x/week to save ₹{save_amt:,.0f}.",
            "saving": round(save_amt, 2)
        })

    # Rule 4: Emergency fund
    if income_total > 0:
        emergency_target = income_total * 6
        recommendations.append({
            "type": "financial_tip",
            "priority": "low",
            "icon": "🏦",
            "title": "Build Emergency Fund",
            "detail": f"Aim for 6 months of expenses (≈₹{expense_total * 6:,.0f}) as an emergency fund. "
                      f"Start by setting aside ₹{income_total * 0.05:,.0f}/month.",
            "saving": 0
        })

    # Rule 5: Investment tip
    recommendations.append({
        "type": "investment",
        "priority": "low",
        "icon": "📈",
        "title": "Start SIP Investment",
        "detail": "Even ₹500/month in a mutual fund SIP at 12% annual return "
                  "grows to ₹1.12 Lakhs in 10 years through compounding.",
        "saving": 0
    })

    # Sort by priority
    priority_order = {"high": 0, "medium": 1, "low": 2}
    recommendations.sort(key=lambda x: priority_order[x["priority"]])

    return {
        "recommendations": recommendations,
        "potential_savings": round(potential_savings, 2),
        "savings_rate": round((income_total - expense_total) / income_total * 100, 1) if income_total > 0 else 0
    }


# ─── 4. Spending Insights (Anomaly Detection + Pattern Analysis) ───────────────

def get_spending_insights(df: pd.DataFrame) -> dict:
    """
    Generates AI-powered insights: anomalies, top categories, spending patterns.
    """
    if df.empty:
        return {"insights": [], "anomalies": []}

    df["date"] = pd.to_datetime(df["date"])
    df["day_of_week"] = df["date"].dt.day_name()
    df["month"] = df["date"].dt.to_period("M")

    expenses = df[df["type"] == "expense"].copy()
    insights = []
    anomalies = []

    if expenses.empty:
        return {"insights": insights, "anomalies": anomalies}

    # 1. Top spending day of week
    day_spend = expenses.groupby("day_of_week")["amount"].sum()
    top_day = day_spend.idxmax()
    insights.append({
        "icon": "📅",
        "title": "Peak Spending Day",
        "text": f"You spend the most on {top_day}s (₹{day_spend[top_day]:,.0f} total). "
                f"Plan purchases on weekdays to reduce impulse buying."
    })

    # 2. Category dominance
    cat_spend = expenses.groupby("category")["amount"].sum()
    top_cat = cat_spend.idxmax()
    top_pct = cat_spend[top_cat] / cat_spend.sum() * 100
    insights.append({
        "icon": "🏆",
        "title": "Dominant Category",
        "text": f"{top_cat} accounts for {top_pct:.1f}% of your total spending. "
                f"Review if this aligns with your priorities."
    })

    # 3. Month-over-month change
    months = sorted(expenses["month"].unique())
    if len(months) >= 2:
        last_m = expenses[expenses["month"] == months[-1]]["amount"].sum()
        prev_m = expenses[expenses["month"] == months[-2]]["amount"].sum()
        change_pct = (last_m - prev_m) / prev_m * 100 if prev_m > 0 else 0
        emoji = "📈" if change_pct > 0 else "📉"
        direction = "increased" if change_pct > 0 else "decreased"
        insights.append({
            "icon": emoji,
            "title": "Month-over-Month Trend",
            "text": f"Spending {direction} by {abs(change_pct):.1f}% compared to last month "
                    f"(₹{last_m:,.0f} vs ₹{prev_m:,.0f})."
        })

    # 4. Anomaly detection (Z-score method)
    cat_stats = expenses.groupby("category")["amount"].agg(["mean", "std"])
    for _, row in expenses.iterrows():
        cat = row.get("category", "Other")
        if cat in cat_stats.index:
            mean = cat_stats.loc[cat, "mean"]
            std = cat_stats.loc[cat, "std"]
            if std > 0:
                z = abs((row["amount"] - mean) / std)
                if z > 2.5:
                    anomalies.append({
                        "date": str(row["date"].date()),
                        "description": row["description"],
                        "amount": row["amount"],
                        "category": cat,
                        "reason": f"Unusually high for {cat} (avg ₹{mean:.0f})"
                    })

    # 5. Savings consistency
    monthly_savings = []
    for m in months[-3:]:
        m_df = df[df["month"] == m]
        inc = m_df[m_df["type"] == "income"]["amount"].sum()
        exp = m_df[m_df["type"] == "expense"]["amount"].sum()
        monthly_savings.append(inc - exp)

    if monthly_savings:
        avg_saving = np.mean(monthly_savings)
        emoji = "✅" if avg_saving > 0 else "❌"
        insights.append({
            "icon": emoji,
            "title": "Savings Consistency",
            "text": f"Average monthly savings over last 3 months: ₹{avg_saving:,.0f}. "
                    + ("Great job maintaining positive savings!" if avg_saving > 0
                       else "Consider reducing expenses to start saving.")
        })

    return {
        "insights": insights,
        "anomalies": anomalies[:5]  # Top 5 anomalies
    }


# ─── 5. AI Finance Chatbot ─────────────────────────────────────────────────────

# def chat_with_ai(message: str, df: pd.DataFrame) -> str:
#     """Rule-based AI chatbot with financial context awareness."""
#     msg = message.lower().strip()
    
#     # Build financial context
#     if not df.empty:
#         df["date"] = pd.to_datetime(df["date"])
#         now = datetime.now()
#         month_start = now.replace(day=1)
        
#         # Get this month's transactions
#         monthly_exp = df[(df["type"] == "expense") & (df["date"] >= month_start)]["amount"]
#         monthly_inc = df[(df["type"] == "income") & (df["date"] >= month_start)]["amount"]
        
#         total_expense = monthly_exp.sum()
#         total_income = monthly_inc.sum()
#         savings = total_income - total_expense
#         savings_rate = (savings / total_income * 100) if total_income > 0 else 0
        
#         # Get category spending
#         cat_spend = df[(df["type"] == "expense") & (df["date"] >= month_start)]\
#             .groupby("category")["amount"].sum().sort_values(ascending=False)
#         top_category = cat_spend.index[0] if not cat_spend.empty else "N/A"
#         top_amount = cat_spend.iloc[0] if not cat_spend.empty else 0
        
#         # Get total transactions count
#         total_txns = len(df)
        
#     else:
#         total_expense = total_income = savings = savings_rate = 0
#         top_category = "N/A"
#         top_amount = 0
#         total_txns = 0
    
#     # ── Intent matching ──
    
#     # Spending summary
#     if any(w in msg for w in ["spend", "spent", "expense", "how much", "total"]):
#         if "category" in msg or any(cat.lower() in msg for cat in cat_spend.index if not df.empty):
#             for cat in (cat_spend.index if not df.empty else []):
#                 if cat.lower() in msg:
#                     return f"You've spent ₹{cat_spend[cat]:,.0f} on {cat} this month. That's {cat_spend[cat]/total_expense*100:.1f}% of your total expenses."
#         return f"This month you've spent ₹{total_expense:,.2f} in total. Your highest spending category is {top_category} at ₹{top_amount:,.0f}."
    
#     # Savings queries
#     elif any(w in msg for w in ["sav", "saving", "save money"]):
#         if savings > 0:
#             return f"You've saved ₹{savings:,.2f} this month — a {savings_rate:.1f}% savings rate. The recommended rate is 20%. " + ("You're doing great! 🎉" if savings_rate >= 20 else f"Try to increase savings by ₹{(total_income*0.2-savings):,.0f} more.")
#         else:
#             return f"This month you're overspending by ₹{abs(savings):,.0f}. Cut discretionary spending in {top_category} to get back on track."
    
#     # Income queries
#     elif any(w in msg for w in ["income", "earn", "salary"]):
#         return f"Your total income this month is ₹{total_income:,.2f}."
    
#     # Budget advice
#     elif any(w in msg for w in ["budget", "plan", "allocat"]):
#         needs = total_income * 0.50
#         wants = total_income * 0.30
#         savings_goal = total_income * 0.20
#         return f"Based on your income of ₹{total_income:,.0f}, the 50-30-20 budget rule suggests:\n• Needs (rent, food, utilities): ₹{needs:,.0f}\n• Wants (entertainment, shopping): ₹{wants:,.0f}\n• Savings & investments: ₹{savings_goal:,.0f}"
    
#     # Investment advice
#     elif any(w in msg for w in ["invest", "mutual fund", "sip", "stock", "fd"]):
#         return "Here are some investment options for beginners:\n• SIP in Mutual Funds: Start from ₹500/month (ELSS for tax saving)\n• PPF: Safe long-term investment with tax benefits (15-year lock-in)\n• Fixed Deposits: 6-7% interest, ideal for emergency fund\n• Index Funds: Low-cost, market-tracking investments\n• NPS: National Pension System for retirement planning\n\n⚠️ This is general info, not financial advice. Consult a SEBI-registered advisor."
    
#     # EMI / loan
#     elif any(w in msg for w in ["emi", "loan", "debt", "credit"]):
#         return f"Managing debt effectively:\n• Keep EMIs under 40% of monthly income\n• Always pay credit card bill in full to avoid 36-48% interest\n• Use the avalanche method: pay highest interest debt first\n• Avoid taking loans for depreciating assets like gadgets\n• With your income of ₹{total_income:,.0f}, max safe EMI is ₹{total_income*0.4:,.0f}"
    
#     # Tax
#     elif any(w in msg for w in ["tax", "itr", "deduction"]):
#         return "Tax-saving tips (India):\n• Section 80C: Save up to ₹1.5L via ELSS, PPF, LIC, ULIP, home loan principal\n• Section 80D: Health insurance premium deduction up to ₹25,000\n• HRA exemption: If paying rent, claim HRA\n• Section 80CCC: NPS contribution up to ₹50,000 extra\n• File ITR by July 31 every year to avoid penalties"
    
#     # Emergency fund
#     elif any(w in msg for w in ["emergency", "fund", "backup"]):
#         target = total_expense * 6
#         return f"Emergency fund target: 6 months of expenses = ₹{target:,.0f}. Keep this in a high-yield savings account or liquid mutual fund. Start by saving 5% of income (₹{total_income*0.05:,.0f}/month) until you reach the target."
    
#     # Greeting
#     elif any(w in msg for w in ["hello", "hi", "hey", "namaste", "hii"]):
#         return f"Hello! 👋 I'm your FinTrack AI assistant.\n\n📊 Your current financial snapshot:\n💰 Income: ₹{total_income:,.0f}\n💸 Expenses: ₹{total_expense:,.0f}\n🏦 Savings: ₹{savings:,.0f} ({savings_rate:.1f}%)\n📝 Total Transactions: {total_txns}\n\nI can help you with:\n• Spending summaries and analysis\n• Savings tips and recommendations\n• Budget planning (50-30-20 rule)\n• Investment advice\n• Tax-saving strategies\n\nWhat would you like to know about your finances?"
    
#     # Help
#     elif any(w in msg for w in ["help", "what can", "feature"]):
#         return "I can answer questions like:\n• 'How much did I spend this month?'\n• 'What are my savings?'\n• 'Give me budget advice'\n• 'How should I invest my money?'\n• 'How can I save on taxes?'\n• 'What is my top spending category?'\n• 'How do I build an emergency fund?'"
    
#     # Top spending category
#     elif any(w in msg for w in ["top", "highest", "most", "biggest"]):
#         return f"Your highest spending category this month is {top_category} at ₹{top_amount:,.0f} ({top_amount/total_expense*100:.1f}% of expenses)."
    
#     # Transactions count
#     elif any(w in msg for w in ["transaction", "entries", "records"]):
#         return f"You have {total_txns} total transactions in your database. This month you've made {len(df[df['date'] >= month_start])} transactions."
    
#     # Default
#     else:
#         return f"Here's your financial snapshot:\n💰 Income: ₹{total_income:,.0f}\n💸 Expenses: ₹{total_expense:,.0f}\n🏦 Savings: ₹{savings:,.0f} ({savings_rate:.1f}%)\n📝 Total Transactions: {total_txns}\n\nTry asking: 'How much did I spend?', 'Give me savings tips', or 'Budget advice'."

def chat_with_ai(message: str, df: pd.DataFrame) -> str:
    """AI Chatbot - Returns a generic response for all messages"""
    return "🚀 This feature is coming soon! We're working hard to bring you an amazing AI assistant. Stay tuned! 🎉"

## 📄 3. Module: report_generator.py

This module generates professional monthly financial reports in PDF format using ReportLab.

In [6]:
"""
FinTrack AI - Monthly Report Generator
Generates a professional PDF financial report using ReportLab.
"""

import os
from datetime import datetime
import pandas as pd

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, HRFlowable
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
    REPORTLAB_AVAILABLE = True
except ImportError:
    REPORTLAB_AVAILABLE = False


def generate_monthly_report(df: pd.DataFrame, df_budgets: pd.DataFrame, month: str) -> str:
    """
    Generate a monthly financial report PDF.
    Falls back to a plain text file if ReportLab is not installed.
    """
    os.makedirs("reports", exist_ok=True)
    path = f"reports/FinTrack_Report_{month}.pdf"

    if not REPORTLAB_AVAILABLE:
        path = path.replace(".pdf", ".txt")
        _generate_text_report(df, df_budgets, month, path)
        return path

    # Filter for requested month
    df["date"] = pd.to_datetime(df["date"])
    year, mon = map(int, month.split("-"))
    mask = (df["date"].dt.year == year) & (df["date"].dt.month == mon)
    month_df = df[mask]

    income_total = month_df[month_df["type"] == "income"]["amount"].sum()
    expense_total = month_df[month_df["type"] == "expense"]["amount"].sum()
    savings = income_total - expense_total
    savings_rate = (savings / income_total * 100) if income_total > 0 else 0

    cat_spend = month_df[month_df["type"] == "expense"].groupby("category")["amount"].sum()

    # ── Build PDF ──
    doc = SimpleDocTemplate(path, pagesize=A4, topMargin=2*cm, bottomMargin=2*cm,
                            leftMargin=2*cm, rightMargin=2*cm)
    styles = getSampleStyleSheet()
    story = []

    # Color palette
    PRIMARY = colors.HexColor("#6366F1")
    SECONDARY = colors.HexColor("#10B981")
    DANGER = colors.HexColor("#EF4444")
    DARK = colors.HexColor("#1E293B")
    LIGHT_BG = colors.HexColor("#F8FAFC")
    NEEELA=colors.HexColor("#1e00c7")

    # Header
    header_style = ParagraphStyle("Header", fontSize=28, textColor=PRIMARY,
                                  spaceAfter=4, alignment=TA_CENTER, fontName="Helvetica-Bold")
    sub_style = ParagraphStyle("Sub", fontSize=12, textColor=DARK,
                               spaceAfter=2, alignment=TA_CENTER, fontName="Helvetica")

    story.append(Spacer(1,20))
    story.append(Paragraph("FinTrack AI", header_style))
    story.append(Spacer(1,12))
    month_name = datetime(year, mon, 1).strftime("%B %Y")
    story.append(Paragraph(f"Monthly Financial Report — {month_name}", sub_style))
    story.append(HRFlowable(width="100%", thickness=2, color=PRIMARY, spaceAfter=20))

    # Summary Cards Table
    section_style = ParagraphStyle("Section", fontSize=14, textColor=PRIMARY,
                                   spaceBefore=16, spaceAfter=8, fontName="Helvetica-Bold")
    story.append(Paragraph("Financial Summary", section_style))

    summary_data = [
        ["Metric", "Amount", "Status"],
        ["Total Income", f"{income_total:,.2f}", "✓"],
        ["Total Expenses", f"{expense_total:,.2f}", "✓"],
        ["Net Savings", f"{savings:,.2f}", "✓" if savings >= 0 else "✗"],
        ["Savings Rate", f"{savings_rate:.1f}%", "✓" if savings_rate >= 20 else "↑ Improve"],
    ]

    t = Table(summary_data, colWidths=[8*cm, 6*cm, 4*cm])
    t.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), PRIMARY),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, 0), 11),
    ("BACKGROUND", (0, 1), (-1, -1), LIGHT_BG),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, LIGHT_BG]),
    ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
    ("FONTSIZE", (0, 1), (-1, -1), 10),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#E2E8F0")),
    ("ALIGN", (1, 0), (-1, -1), "CENTER"),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("PADDING", (0, 0), (-1, -1), 8),
    # Color Income row (row index 1) green
    ("TEXTCOLOR", (1, 1), (1, 1), NEEELA),
    # Color Expenses row (row index 2) red
    ("TEXTCOLOR", (1, 2), (1, 2), DANGER),
    # Color Savings row based on value (row index 3)
    ("TEXTCOLOR", (1, 3), (1, 3), SECONDARY if savings >= 0 else DANGER),
]))
    story.append(t)
    story.append(Spacer(1, 16))

    # Category Breakdown
    story.append(Paragraph("Spending by Category", section_style))

    if not cat_spend.empty:
        cat_data = [["Category", "Amount (INR)", "% of Expenses"]]
        for cat, amt in cat_spend.sort_values(ascending=False).items():
            pct = (amt / expense_total * 100) if expense_total > 0 else 0
            cat_data.append([cat, f"{amt:,.2f}", f"{pct:.1f}%"])

        ct = Table(cat_data, colWidths=[8*cm, 6*cm, 4*cm])
        ct.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#6366F1")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, LIGHT_BG]),
            ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
            ("FONTSIZE", (0, 0), (-1, -1), 10),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#E2E8F0")),
            ("ALIGN", (1, 0), (-1, -1), "CENTER"),
            ("PADDING", (0, 0), (-1, -1), 7),
        ]))
        story.append(ct)

    story.append(Spacer(1, 16))

    # Budget vs Actual
    if not df_budgets.empty:
        story.append(Paragraph("Budget vs Actual", section_style))
        bud_data = [["Category", "Budget (INR)", "Actual (INR)", "Status"]]
        for _, row in df_budgets.iterrows():
            actual = cat_spend.get(row["category"], 0)
            status = "✓ Under" if actual <= row["monthly_limit"] else "✗ Over"
            bud_data.append([row["category"], f"{row['monthly_limit']:,.0f}",
                             f"{actual:,.0f}", status])

        bt = Table(bud_data, colWidths=[6*cm, 4*cm, 4*cm, 4*cm])
        bt.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), PRIMARY),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, LIGHT_BG]),
            ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
            ("FONTSIZE", (0, 0), (-1, -1), 10),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#E2E8F0")),
            ("ALIGN", (1, 0), (-1, -1), "CENTER"),
            ("PADDING", (0, 0), (-1, -1), 7),
        ]))
        story.append(bt)

    # Recent Transactions
    story.append(Spacer(1, 16))
    story.append(Paragraph("Top Transactions This Month", section_style))
    top_txns = month_df[month_df["type"] == "expense"].nlargest(10, "amount")
    if not top_txns.empty:
        txn_data = [["Date", "Description", "Category", "Amount (INR)"]]
        for _, row in top_txns.iterrows():
            txn_data.append([
                str(row["date"].date()),
                row["description"][:30],
                row.get("category", "Other"),
                f"{row['amount']:,.2f}"
            ])
        tt = Table(txn_data, colWidths=[4*cm, 7*cm, 5*cm, 4*cm])
        tt.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), PRIMARY),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, LIGHT_BG]),
            ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
            ("FONTSIZE", (0, 0), (-1, -1), 9),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#E2E8F0")),
            ("ALIGN", (3, 0), (3, -1), "RIGHT"),
            ("PADDING", (0, 0), (-1, -1), 6),
        ]))
        story.append(tt)

    # Footer
    story.append(Spacer(1, 30))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#E2E8F0")))
    footer_style = ParagraphStyle("Footer", fontSize=8, textColor=colors.grey,
                                  alignment=TA_CENTER, spaceBefore=8)
    story.append(Paragraph(
        f"Generated by FinTrack AI | {datetime.now().strftime('%d %b %Y, %H:%M')} | For personal use only",
        footer_style
    ))

    doc.build(story)
    return path


def _generate_text_report(df, df_budgets, month, path):
    """Fallback text report if ReportLab not installed."""
    df["date"] = pd.to_datetime(df["date"])
    year, mon = map(int, month.split("-"))
    mask = (df["date"].dt.year == year) & (df["date"].dt.month == mon)
    month_df = df[mask]

    income = month_df[month_df["type"] == "income"]["amount"].sum()
    expenses = month_df[month_df["type"] == "expense"]["amount"].sum()
    savings = income - expenses

    lines = [
        "=" * 50,
        "  FinTrack AI - Monthly Financial Report",
        f"  {datetime(year, mon, 1).strftime('%B %Y')}",
        "=" * 50,
        f"\nTotal Income:   ₹{income:,.2f}",
        f"Total Expenses: ₹{expenses:,.2f}",
        f"Net Savings:    ₹{savings:,.2f}",
        f"Savings Rate:   {(savings/income*100):.1f}%\n",
        "\nSpending by Category:",
        "-" * 30,
    ]

    cat_spend = month_df[month_df["type"] == "expense"].groupby("category")["amount"].sum()
    for cat, amt in cat_spend.sort_values(ascending=False).items():
        lines.append(f"  {cat:<20} ₹{amt:>10,.2f}")

    lines += ["\n" + "=" * 50, "Generated by FinTrack AI"]

    with open(path, "w") as f:
        f.write("\n".join(lines))

## 🚀 4. Module: app.py – Complete Flask Application

This is the main Flask application that ties everything together.

In [7]:
"""
FinTrack AI - AI-Powered Personal Finance Manager
Main Flask Application
"""

from flask import Flask, render_template, request, jsonify, send_file
from flask_cors import CORS
import sqlite3
import json
import os
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from ml_engine import (
    categorize_transaction,
    predict_future_expenses,
    generate_savings_recommendations,
    get_spending_insights,
    chat_with_ai
)
from report_generator import generate_monthly_report
import warnings
warnings.filterwarnings('ignore')

app = Flask(__name__)
CORS(app)

DB_PATH = "data/fintrack.db"


# ─── Database Setup ────────────────────────────────────────────────────────────

def init_db():
    os.makedirs("data", exist_ok=True)
    os.makedirs("reports", exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""
        CREATE TABLE IF NOT EXISTS transactions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            date TEXT NOT NULL,
            description TEXT NOT NULL,
            amount REAL NOT NULL,
            type TEXT NOT NULL,
            category TEXT,
            notes TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    c.execute("""
        CREATE TABLE IF NOT EXISTS budgets (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            category TEXT UNIQUE NOT NULL,
            monthly_limit REAL NOT NULL
        )
    """)
    c.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            role TEXT NOT NULL,
            message TEXT NOT NULL,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()

    # Seed with sample data if empty
    c.execute("SELECT COUNT(*) FROM transactions")
    if c.fetchone()[0] == 0:
        _seed_sample_data(c)
        conn.commit()

    conn.close()


def _seed_sample_data(c):
    """Insert realistic sample transactions for demo."""
    import random
    random.seed(42)

    categories = {
        "Food & Dining": ["Zomato Order", "McDonald's", "Swiggy", "Local Restaurant", "Cafe Coffee Day", "Domino's Pizza"],
        "Shopping": ["Amazon Purchase", "Flipkart Order", "Myntra", "Local Market", "Big Bazaar"],
        "Transportation": ["Ola Cab", "Uber Ride", "Metro Card Recharge", "Bus Pass", "Petrol"],
        "Entertainment": ["Netflix", "Spotify", "Movie Ticket", "BookMyShow", "Prime Video"],
        "Utilities": ["Electricity Bill", "Water Bill", "Internet Bill", "Mobile Recharge"],
        "Healthcare": ["Pharmacy", "Doctor Consultation", "Lab Test", "Medicine"],
        "Education": ["Course Fee", "Books", "Stationery", "Online Course"],
    }

    incomes = [
        ("Salary Credit", 45000),
        ("Freelance Payment", 8000),
        ("Part-time Work", 5000),
    ]

    today = datetime.now()
    transactions = []

    for month_offset in range(6):
        base_date = today - timedelta(days=30 * month_offset)

        # Monthly income
        for desc, base_amount in incomes:
            if random.random() > 0.3:
                amt = base_amount + random.randint(-500, 500)
                dt = (base_date.replace(day=1) + timedelta(days=random.randint(0, 5))).strftime("%Y-%m-%d")
                transactions.append((dt, desc, amt, "income", "Income", ""))

        # Monthly expenses
        for cat, descs in categories.items():
            num_txns = random.randint(2, 6)
            for _ in range(num_txns):
                desc = random.choice(descs)
                amt = round(random.uniform(100, 3000), 2)
                day = random.randint(1, 28)
                try:
                    dt = base_date.replace(day=day).strftime("%Y-%m-%d")
                except ValueError:
                    dt = base_date.strftime("%Y-%m-%d")
                transactions.append((dt, desc, amt, "expense", cat, ""))

    c.executemany(
        "INSERT INTO transactions (date, description, amount, type, category, notes) VALUES (?,?,?,?,?,?)",
        transactions
    )

    # Seed budgets
    budgets = [
        ("Food & Dining", 5000),
        ("Shopping", 4000),
        ("Transportation", 3000),
        ("Entertainment", 2000),
        ("Utilities", 2500),
        ("Healthcare", 1500),
        ("Education", 3000),
    ]
    c.executemany("INSERT OR IGNORE INTO budgets (category, monthly_limit) VALUES (?,?)", budgets)


def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


# ─── Routes ────────────────────────────────────────────────────────────────────

@app.route("/")
def index():
    return render_template("index.html")


@app.route("/api/dashboard", methods=["GET"])
def dashboard():
    conn = get_db()
    c = conn.cursor()

    now = datetime.now()
    month_start = now.replace(day=1).strftime("%Y-%m-%d")
    month_end = now.strftime("%Y-%m-%d")

    # This month totals
    c.execute("SELECT SUM(amount) FROM transactions WHERE type='income' AND date >= ?", (month_start,))
    income = c.fetchone()[0] or 0

    c.execute("SELECT SUM(amount) FROM transactions WHERE type='expense' AND date >= ?", (month_start,))
    expenses = c.fetchone()[0] or 0

    # Category breakdown this month
    c.execute("""
        SELECT category, SUM(amount) as total
        FROM transactions WHERE type='expense' AND date >= ?
        GROUP BY category ORDER BY total DESC
    """, (month_start,))
    categories = [{"category": r[0], "total": round(r[1], 2)} for r in c.fetchall()]

    # Monthly trend (last 6 months)
    monthly_data = []
    for i in range(5, -1, -1):
        d = now - timedelta(days=30 * i)
        m_start = d.replace(day=1).strftime("%Y-%m-%d")
        if i > 0:
            d2 = now - timedelta(days=30 * (i - 1))
            m_end = d2.replace(day=1).strftime("%Y-%m-%d")
        else:
            m_end = now.strftime("%Y-%m-%d")

        c.execute("SELECT SUM(amount) FROM transactions WHERE type='income' AND date >= ? AND date <= ?", (m_start, m_end))
        inc = c.fetchone()[0] or 0
        c.execute("SELECT SUM(amount) FROM transactions WHERE type='expense' AND date >= ? AND date <= ?", (m_start, m_end))
        exp = c.fetchone()[0] or 0
        monthly_data.append({
            "month": d.strftime("%b %Y"),
            "income": round(inc, 2),
            "expenses": round(exp, 2)
        })

    # Recent transactions
    c.execute("""
        SELECT id, date, description, amount, type, category
        FROM transactions ORDER BY date DESC, id DESC LIMIT 10
    """)
    recent = [dict(r) for r in c.fetchall()]

    # Budgets
    c.execute("SELECT category, monthly_limit FROM budgets")
    budgets_raw = c.fetchall()
    budgets = []
    for b in budgets_raw:
        c.execute("SELECT SUM(amount) FROM transactions WHERE category=? AND date >= ?",
                  (b[0], month_start))
        spent = c.fetchone()[0] or 0
        budgets.append({
            "category": b[0],
            "limit": b[1],
            "spent": round(spent, 2),
            "pct": round(min(spent / b[1] * 100, 100), 1) if b[1] > 0 else 0
        })

    conn.close()

    return jsonify({
        "income": round(income, 2),
        "expenses": round(expenses, 2),
        "savings": round(income - expenses, 2),
        "savings_rate": round((income - expenses) / income * 100, 1) if income > 0 else 0,
        "categories": categories,
        "monthly_trend": monthly_data,
        "recent_transactions": recent,
        "budgets": budgets
    })


@app.route("/api/transactions", methods=["GET"])
def get_transactions():
    conn = get_db()
    c = conn.cursor()
    page = int(request.args.get("page", 1))
    per_page = int(request.args.get("per_page", 20))
    search = request.args.get("search", "")
    txn_type = request.args.get("type", "")
    category = request.args.get("category", "")
    offset = (page - 1) * per_page

    conditions = []
    params = []
    if search:
        conditions.append("description LIKE ?")
        params.append(f"%{search}%")
    if txn_type:
        conditions.append("type = ?")
        params.append(txn_type)
    if category:
        conditions.append("category = ?")
        params.append(category)

    where = ("WHERE " + " AND ".join(conditions)) if conditions else ""

    c.execute(f"SELECT COUNT(*) FROM transactions {where}", params)
    total = c.fetchone()[0]

    c.execute(
        f"SELECT * FROM transactions {where} ORDER BY date DESC, id DESC LIMIT ? OFFSET ?",
        params + [per_page, offset]
    )
    rows = [dict(r) for r in c.fetchall()]
    conn.close()

    return jsonify({"transactions": rows, "total": total, "page": page, "per_page": per_page})


@app.route("/api/transactions", methods=["POST"])
def add_transaction():
    data = request.json
    description = data.get("description", "").strip()
    amount = float(data.get("amount", 0))
    txn_type = data.get("type", "expense")
    date = data.get("date", datetime.now().strftime("%Y-%m-%d"))
    notes = data.get("notes", "")

    # AI categorization
    category = data.get("category") or (
        categorize_transaction(description, amount, txn_type)
        if txn_type == "expense" else "Income"
    )

    conn = get_db()
    c = conn.cursor()
    c.execute(
        "INSERT INTO transactions (date, description, amount, type, category, notes) VALUES (?,?,?,?,?,?)",
        (date, description, amount, txn_type, category, notes)
    )
    new_id = c.lastrowid
    conn.commit()
    conn.close()

    return jsonify({"success": True, "id": new_id, "category": category, "message": f"Transaction added under '{category}'"})


@app.route("/api/transactions/<int:txn_id>", methods=["DELETE"])
def delete_transaction(txn_id):
    conn = get_db()
    conn.execute("DELETE FROM transactions WHERE id=?", (txn_id,))
    conn.commit()
    conn.close()
    return jsonify({"success": True})


@app.route("/api/predict", methods=["GET"])
def predict():
    conn = get_db()
    df = pd.read_sql_query(
        "SELECT date, amount, category FROM transactions WHERE type='expense'",
        conn
    )
    conn.close()

    predictions = predict_future_expenses(df)
    return jsonify(predictions)


@app.route("/api/recommendations", methods=["GET"])
def recommendations():
    conn = get_db()
    df_txn = pd.read_sql_query("SELECT * FROM transactions", conn)
    df_budgets = pd.read_sql_query("SELECT * FROM budgets", conn)
    conn.close()

    recs = generate_savings_recommendations(df_txn, df_budgets)
    return jsonify(recs)


@app.route("/api/insights", methods=["GET"])
def insights():
    conn = get_db()
    df = pd.read_sql_query("SELECT * FROM transactions", conn)
    conn.close()

    result = get_spending_insights(df)
    return jsonify(result)


@app.route("/api/chat", methods=["POST"])
def chat():
    data = request.json
    user_message = data.get("message", "")
    conn = get_db()
    df = pd.read_sql_query("SELECT * FROM transactions", conn)
    conn.close()
    conn = get_db()
    conn.execute("INSERT INTO chat_history (role, message) VALUES (?, ?)", ("user", user_message))
    conn.commit()
    response = chat_with_ai(user_message, df)
    conn.execute("INSERT INTO chat_history (role, message) VALUES (?, ?)", ("assistant", response))
    conn.commit()
    conn.close()
    return jsonify({"response": response})


@app.route("/api/report", methods=["GET"])
def monthly_report():
    month = request.args.get("month", datetime.now().strftime("%Y-%m"))
    conn = get_db()
    df = pd.read_sql_query("SELECT * FROM transactions", conn)
    df_budgets = pd.read_sql_query("SELECT * FROM budgets", conn)
    conn.close()

    path = generate_monthly_report(df, df_budgets, month)
    return send_file(path, as_attachment=True, download_name=f"FinTrack_Report_{month}.pdf")


@app.route("/api/budgets", methods=["GET"])
def get_budgets():
    conn = get_db()
    c = conn.cursor()
    
    # Get current month start date
    now = datetime.now()
    month_start = now.replace(day=1).strftime("%Y-%m-%d")
    
    # Get all budgets
    c.execute("SELECT * FROM budgets")
    rows = c.fetchall()
    
    budgets = []
    for row in rows:
        # Calculate spent amount for this category this month
        c.execute(
            "SELECT SUM(amount) FROM transactions WHERE category=? AND type='expense' AND date >= ?",
            (row["category"], month_start)
        )
        spent = c.fetchone()[0] or 0
        
        budgets.append({
            "category": row["category"],
            "limit": row["monthly_limit"],
            "spent": round(spent, 2),
            "pct": round(min(spent / row["monthly_limit"] * 100, 100), 1) if row["monthly_limit"] > 0 else 0
        })
    
    conn.close()
    return jsonify(budgets)


@app.route("/api/budgets", methods=["POST"])
def set_budget():
    data = request.json
    conn = get_db()
    conn.execute(
        "INSERT OR REPLACE INTO budgets (category, monthly_limit) VALUES (?,?)",
        (data["category"], float(data["monthly_limit"]))
    )
    conn.commit()
    conn.close()
    return jsonify({"success": True})


@app.route("/api/categories", methods=["GET"])
def get_categories():
    categories = [
        "Food & Dining", "Shopping", "Transportation", "Entertainment",
        "Utilities", "Healthcare", "Education", "Travel", "Other"
    ]
    return jsonify(categories)


if __name__ == "__main__":
    init_db()
    print("\n" + "="*55)
    print("  🚀  FinTrack AI — Personal Finance Manager")
    print("="*55)
    print("  Running at: http://127.0.0.1:5000")
    print("="*55 + "\n")
    app.run(debug=True, port=5000)

@app.route("/reports")
def list_reports():
    """List all generated reports"""
    import os
    reports_dir = "reports"
    if not os.path.exists(reports_dir):
        return jsonify([])
    
    files = []
    for f in os.listdir(reports_dir):
        if f.endswith('.pdf') or f.endswith('.txt'):
            files.append(f)
    
    return jsonify(sorted(files, reverse=True))

@app.route('/reports/<filename>')
def download_report(filename):
    """Download a report file"""
    import os
    from flask import send_from_directory
    return send_from_directory('reports', filename)


  🚀  FinTrack AI — Personal Finance Manager
  Running at: http://127.0.0.1:5000

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

## 🏃 5. Run the Complete Application

Run this cell to start the Flask server.

In [ ]:
# Start the Flask server
if __name__ == "__main__":
    init_db()
    app.run(debug=False,use_reloader=False, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Jul/2026 16:00:47] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [02/Jul/2026 16:00:47] "GET /static/css/style.css HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:47] "GET /static/css/dashboard.css HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:48] "GET /static/js/utils.js HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:48] "GET /static/js/api.js HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:48] "GET /static/js/app.js HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:48] "GET /static/js/dashboard.js HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:48] "GET /static/js/transactions.js HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:48] "GET /static/js/budgets.js HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:48] "GET /static/js/analytics.js HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:48] "GET /static/js/chat.js HTTP/1.1" 304 -
127.0.0.1 - - [02/Jul/2026 16:00:48] "GET /static/js/reports.js HTTP/1.1" 304 -
127

## 🌐 6. Access the Application

Once the server is running, open your browser and go to:

**👉 http://127.0.0.1:5000**

### Features:
- 📊 **Dashboard** – See your financial summary, charts, and budgets
- 📝 **Transactions** – Add, view, search, and delete transactions
- 💰 **Budgets** – Set and track monthly spending limits
- 📈 **Analytics** – Get spending insights, predictions, and recommendations
- 🤖 **AI Chat** – Ask questions about your finances
- 📄 **Reports** – Generate monthly PDF reports

## 📋 7. API Endpoints Reference

| Method | Endpoint | Description |
|--------|----------|-------------|
| GET | `/api/dashboard` | Get dashboard summary |
| GET | `/api/transactions` | List transactions (paginated) |
| POST | `/api/transactions` | Add a new transaction |
| DELETE | `/api/transactions/<id>` | Delete a transaction |
| GET | `/api/budgets` | List all budgets |
| POST | `/api/budgets` | Set/update a budget |
| GET | `/api/predict` | Get expense predictions |
| GET | `/api/recommendations` | Get savings recommendations |
| GET | `/api/insights` | Get spending insights |
| POST | `/api/chat` | Send a message to AI |
| GET | `/api/report` | Download monthly report PDF |
| GET | `/api/categories` | List all categories |

## ✅ 8. Testing the Application

Run these commands in a separate cell or terminal to test the API:

In [ ]:
# Test the dashboard endpoint
import requests
import json

try:
    response = requests.get("http://127.0.0.1:5000/api/dashboard")
    if response.status_code == 200:
        data = response.json()
        print("✅ Dashboard API is working!")
        print(f"   Income: ₹{data['income']:,.2f}")
        print(f"   Expenses: ₹{data['expenses']:,.2f}")
        print(f"   Savings: ₹{data['savings']:,.2f}")
        print(f"   Savings Rate: {data['savings_rate']}%")
    else:
        print("❌ Dashboard API returned:", response.status_code)
except Exception as e:
    print("❌ Could not connect to server. Make sure Flask is running.")
    print(f"   Error: {e}")

❌ Could not connect to server. Make sure Flask is running.
   Error: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/dashboard (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))


## 🎉 9. Done!

Your FinTrack AI application is now fully set up and running.

**Open in browser:** http://127.0.0.1:5000

**Project Structure:**
```
project/
├── app.py              # Flask server
├── ml_engine.py        # AI/ML logic
├── report_generator.py # PDF reports
├── templates/
│   └── index.html      # Frontend
├── static/
│   ├── css/            # Stylesheets
│   └── js/             # JavaScript files
├── data/
│   └── fintrack.db     # SQLite database
└── reports/            # Generated PDF reports
```